# Vision Assignment P05 — LeNet-5 dan AlexNet

Implementasi model CNN untuk MNIST dan CIFAR-10 serta eksperimen learning rate dan batch size.

In [ ]:
%pip install -q torch torchvision matplotlib

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
class LeNet5(nn.Module):
    def __init__(self, classes=10):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1,6,5), nn.Tanh(), nn.AvgPool2d(2), nn.Conv2d(6,16,5), nn.Tanh(), nn.AvgPool2d(2), nn.Conv2d(16,120,5), nn.Tanh())
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(120,84), nn.Tanh(), nn.Linear(84,classes))
    def forward(self, x): return self.classifier(self.features(x))

class AlexNetCIFAR(nn.Module):
    def __init__(self, classes=10):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(3,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(64,192,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(192,384,3,padding=1),nn.ReLU(),nn.Conv2d(384,256,3,padding=1),nn.ReLU(),nn.Conv2d(256,256,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.classifier = nn.Sequential(nn.Flatten(),nn.Dropout(.5),nn.Linear(256*4*4,1024),nn.ReLU(),nn.Dropout(.5),nn.Linear(1024,classes))
    def forward(self, x): return self.classifier(self.features(x))

print('LeNet parameters:', sum(p.numel() for p in LeNet5().parameters()))
print('AlexNet-CIFAR parameters:', sum(p.numel() for p in AlexNetCIFAR().parameters()))

In [ ]:
mnist_tf = transforms.Compose([transforms.Resize((32,32)), transforms.ToTensor(), transforms.Normalize((.1307,),(.3081,))])
mnist_train = datasets.MNIST('data', train=True, download=True, transform=mnist_tf)
mnist_test = datasets.MNIST('data', train=False, download=True, transform=mnist_tf)
train_loader = DataLoader(mnist_train, batch_size=128, shuffle=True)
test_loader = DataLoader(mnist_test, batch_size=256)

def run_epoch(model, loader, criterion, optimizer=None):
    model.train(optimizer is not None); loss_sum=correct=total=0
    for x,y in loader:
        x,y=x.to(device),y.to(device)
        if optimizer: optimizer.zero_grad()
        out=model(x); loss=criterion(out,y)
        if optimizer: loss.backward(); optimizer.step()
        loss_sum += loss.item()*len(y); correct += (out.argmax(1)==y).sum().item(); total += len(y)
    return loss_sum/total, correct/total

model=LeNet5().to(device); criterion=nn.CrossEntropyLoss(); optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
for epoch in range(2):
    tr=run_epoch(model,train_loader,criterion,optimizer); te=run_epoch(model,test_loader,criterion)
    print(f'Epoch {epoch+1}: train loss={tr[0]:.4f}, train acc={tr[1]:.3f}, test loss={te[0]:.4f}, test acc={te[1]:.3f}')

## Eksperimen

Ulangi training dengan learning rate `1e-2`, `1e-3`, `1e-4` dan batch size `64`, `128`, `256`. Catat loss, akurasi, dan waktu per epoch. Learning rate terlalu besar cenderung tidak stabil; terlalu kecil memperlambat konvergensi.

## Penjelasan Konsep dan Eksperimen

CNN menggunakan receptive field lokal dan weight sharing sehingga lebih efisien daripada MLP. LeNet-5 memakai convolution, average pooling, dan fully connected layer untuk digit grayscale. AlexNet memperkenalkan ReLU, dropout, augmentasi data, dan training GPU; pada notebook ini arsitekturnya diadaptasi untuk CIFAR-10 32×32.

Ulangi training dengan learning rate `1e-2`, `1e-3`, `1e-4` dan batch size `64`, `128`, `256`. Catat loss, akurasi, waktu training, serta gap train–test sebagai indikasi overfitting. Gunakan `torch.manual_seed(42)` agar perbandingan reproducible.

LeNet-5 sesuai untuk MNIST karena input kecil dan grayscale. AlexNet perlu diadaptasi untuk CIFAR-10 dengan kernel dan fully connected layer yang sesuai ukuran 32×32.

In [ ]:
# Kesimpulan otomatis dari epoch terakhir
print('ANALISIS OTOMATIS')
print(f'- Train accuracy: {tr[1]:.3f}')
print(f'- Test accuracy: {te[1]:.3f}')
gap = tr[1] - te[1]
print(f'- Generalization gap: {gap:.3f}')
print('KESIMPULAN:', 'Ada indikasi overfitting.' if gap > 0.10 else 'Gap train-test masih terkendali pada eksperimen ini.')